# ClinRAG: Clinical Knowledge Assistant #



## RAG-Based Medical Question Answering System ##

In [1]:
#! pip install langchain-community

In [2]:
#! pip install tqdm

In [3]:
# Load the documents

from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader('datafiles', glob="*.txt", show_progress=True, loader_cls=TextLoader)

kb_docs = loader.load()

D:\AITraining\Project2RAG\env_RAGProject\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|███████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 495.84it/s]


In [4]:
# Chunk the loaded documents

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks = text_splitter.split_documents(kb_docs)

print("Number of Documents:", len(kb_docs))
print()
print("Number of Chunks:", len(chunks))

Number of Documents: 3

Number of Chunks: 89


In [5]:
#! pip install langchain-chroma
#! pip install langchain-openai

In [6]:
#!pip install sentence-transformers

In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\dkotha1\AppData\Local\Temp\ipykernel_23120\384917042.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 345.53it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# Initialize a ChromaDB Connection
from langchain_chroma import Chroma

# Initialize the database connection
# If database exist, it will connect with the collection_name and persist_directory
# Otherwise a new collection will be created
db = Chroma(collection_name="vector_database", 
            embedding_function=embedding_model, 
            persist_directory="./chroma_db_")

# Insert chunks in the vector database
db.add_documents(documents=chunks)

# We can check the already existing values
print(len(db.get()["ids"]))

356


## **Building an End-to-End RAG Chain**

**Step 1: Initialize the Chroma DB Connection**  
**Step 2: Create a Retriever Object**   
**Step 3: Initialize a Chat Prompt Template**  
**Step 4: Initialize a Generator (i.e. Chat Model)**  
**Step 5: Initialize a Output Parser**   
**Step 6: Define a RAG Chain**  
**Step 7: Invoke the Chain**

In [9]:
# Step 1: Initialize the Chroma DB Connection

from langchain_chroma import Chroma

# Initialize the database connection
# If database exist, it will connect with the collection_name and persist_directory
# Otherwise a new collection will be created
db = Chroma(collection_name="vector_database", 
            embedding_function=embedding_model, 
            persist_directory="./chroma_db_")

# We can check the already existing values
print(len(db.get()["ids"]))

356


In [10]:
# Step 2: Create a Retriever Object 

retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [11]:
# Step 3: Initialize a Chat Prompt Template

from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
    ]
)

prompt_template

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the following context:\n{context}\nAnswer the question based on the above context: {question}.\nProvide a detailed answer.\nDon’t justify your answers.\nDon’t give information not mentioned in the CONTEXT INFORMATION.\nDo not say "according to the context" or "mentioned in the context" or similar.\n'), additional_kwargs={})])

In [12]:
#!pip install langchain-groq

In [13]:
# Step 4: Initialize a Generator (Groq Model)

from langchain_groq import ChatGroq

f = open('keys/groqKey.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    groq_api_key=GROQ_API_KEY,   # paste your Groq key here
    model_name="llama-3.1-8b-instant",        # recommended model
    temperature=0                      # keeps answers factual
)

In [14]:
# Step 5: Initialize a Output Parser

from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [15]:
generator_chain = prompt_template | chat_model | parser

In [16]:
# Helper function to join the retrieved chunks

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [17]:
# Step 6: Define a RAG Chain

from langchain_core.runnables import RunnablePassthrough

rag_chain = {
    "context": retriever | format_docs, 
    "question": RunnablePassthrough()
} | generator_chain

In [18]:
# Invoke the Chain
query = 'What is Alzheimer’s disease?'
response = rag_chain.invoke(query)
print(response)

Alzheimer's disease is a neurodegenerative disease. 
It is the most common form of dementia, accounting for around 60–70% of cases. 
The most common early symptom is difficulty in remembering recent events. 
As the disease advances, symptoms can include problems with language, disorientation (including easily getting lost), mood swings, loss of motivation, self-neglect, and behavioral issues. 
As a person's condition declines, they often withdraw from family and society. 
Gradually, bodily functions are lost, ultimately leading to death. 
Although the speed of progression can vary, the average life expectancy following diagnosis is three to twelve years. 
The causes of Alzheimer's disease remain poorly understood. 
There are many environmental and genetic risk factors associated with its development.


In [19]:
query = 'What is typically the earliest symptom of Alzheimer’s disease?'
response = rag_chain.invoke(query)
print(response)

The earliest symptom of Alzheimer's disease is short term memory loss. 

It shows up as difficulty in remembering recently learned facts and inability to acquire new information.


In [20]:
query = ' What is the most common cause of dementia?'
response = rag_chain.invoke(query)
print(response)

The pathology of Alzheimer's disease is accompanied by lesions that are characteristic of other brain disorders in more than half of the cases examined neuropathologically, and especially in very old people.

The most common of these comorbid conditions are vascular disease, Lewy body disease, and TDP-43 proteinopathy.

However, the question asks for the most common cause of dementia. 

Alzheimer's disease is not explicitly stated as the most common cause of dementia, but it is implied that it is a significant contributor to dementia, as it is the pathology being examined.

Therefore, the most common cause of dementia is Alzheimer's disease, accompanied by vascular disease, Lewy body disease, and TDP-43 proteinopathy in more than half of the cases.


In [22]:
query = 'Is there a cure for Alzheimer’s disease?'
rag_chain.invoke(query)


"There is no cure for Alzheimer's disease."